# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [3]:
from pathlib import Path
import duckdb
import polars as pl

# Cari folder project secara otomatis
current_dir = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        p for p in [current_dir, *current_dir.parents]
        if (p / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Folder data/raw tidak ditemukan.")

DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_FILES = sorted(DATA_DIR.glob("mc4_id_*.parquet"))

if not DATA_FILES:
    raise FileNotFoundError(f"Dataset tidak ditemukan di {DATA_DIR}")

OUTPUT_PATH = (
    PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"
)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Lokasi project:", PROJECT_ROOT)
print("\nFile dataset:")

for file in DATA_FILES:
    print(
        f"- {file.name}: "
        f"{file.stat().st_size / 1_000_000:.2f} MB"
    )

# Lazy scan: belum memuat seluruh dataset ke RAM
lf = pl.scan_parquet([str(file) for file in DATA_FILES])

print("\nStruktur kolom:")
print(lf.collect_schema())

# Hitung baris menggunakan streaming
jumlah_baris = lf.select(
    pl.len().alias("jumlah_baris")
).collect(engine="streaming")

print("\nJumlah baris:")
display(jumlah_baris)

# Tampilkan lima baris saja
print("\nContoh data:")
display(lf.head(5).collect())

Lokasi project: /home/jovyan/work

File dataset:
- mc4_id_0000.parquet: 289.05 MB
- mc4_id_0001.parquet: 291.06 MB

Struktur kolom:
Schema({'text': String, 'timestamp': String, 'url': String})

Jumlah baris:


jumlah_baris
u32
305000



Contoh data:


text,timestamp,url
str,str,str
"""Rentetan Program Kegiatan Prud…","""2020-01-23T09:19:49Z""","""https://www.prudential.co.id/i…"
"""TRISHCO (Trisakti Hospitality …","""2019-11-21T05:28:36Z""","""http://stptrisakti.ac.id/trish…"
"""Rapat Koordinasi Lintas Sektor…","""2020-02-22T23:02:46Z""","""https://wartanusa.id/rapat-koo…"
"""Nusa Penida Festival 2019, Kem…","""2020-06-02T18:13:32Z""","""http://metrobali.com/festival-…"
"""Wisata Buatan Archives - Dinas…","""2020-04-07T16:06:04Z""","""https://visitingjogja.com/info…"


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [4]:
con = duckdb.connect()

# Daftarkan kedua file Parquet sebagai sumber query
file_list_sql = ", ".join(
    "'" + str(file).replace("'", "''") + "'"
    for file in DATA_FILES
)

query = f"""
    SUMMARIZE
    SELECT *
    FROM read_parquet([{file_list_sql}])
"""

profil = con.execute(query).df()

display(profil)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,text,VARCHAR,""" /> Cegah Perkembangbiakan Anopheles, Satgas ...",🥇 Kursus privat Mengaji di Padang | Superprof\...,321120,<NA>,<NA>,<NA>,<NA>,<NA>,305000,0.0
1,timestamp,VARCHAR,2013-05-18T05:01:28Z,2020-08-15T21:24:00Z,274561,<NA>,<NA>,<NA>,<NA>,<NA>,305000,0.0
2,url,VARCHAR,http://01detail.com/bandarq-terpercaya-memilik...,https://zurrahmah.wordpress.com/2015/04/09/lan...,307602,<NA>,<NA>,<NA>,<NA>,<NA>,305000,0.0


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [5]:
# Deteksi missing values secara hemat memori
kolom = lf.collect_schema().names()

# Hitung jumlah baris
jumlah_baris = (
    lf.select(pl.len())
    .collect(engine="streaming")
    .item()
)

# Hitung nilai null setiap kolom
null_counts = (
    lf.select([
        pl.col(c).null_count().alias(c)
        for c in kolom
    ])
    .collect(engine="streaming")
)

# Susun hasil menjadi tabel
jumlah_null = [
    null_counts[c][0]
    for c in kolom
]

null_summary = pl.DataFrame({
    "kolom": kolom,
    "jumlah_null": jumlah_null,
})

null_summary = null_summary.with_columns(
    (
        pl.col("jumlah_null") / jumlah_baris * 100
    ).round(2).alias("persen_null")
)

print(f"Total baris: {jumlah_baris:,}")
print("Ringkasan missing values:")
display(null_summary)

Total baris: 305,000
Ringkasan missing values:


kolom,jumlah_null,persen_null
str,i64,f64
"""text""",0,0.0
"""timestamp""",0,0.0
"""url""",0,0.0


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [6]:
# Penanganan missing values untuk dataset teks mC4

# Rapikan teks, URL, dan konversi timestamp
lf_missing_clean = (
    lf.with_columns(
        pl.col("text")
        .str.strip_chars()
        .alias("text"),

        pl.col("timestamp")
        .str.strip_chars()
        .str.to_datetime(
            strict=False,
            time_zone="UTC"
        )
        .alias("timestamp"),

        pl.col("url")
        .str.strip_chars()
        .alias("url"),
    )
    .filter(
        # Teks wajib tersedia dan tidak kosong
        pl.col("text").is_not_null()
        & (pl.col("text") != "")

        # Timestamp harus berhasil dikonversi
        & pl.col("timestamp").is_not_null()

        # URL wajib tersedia dan menggunakan HTTP/HTTPS
        & pl.col("url").is_not_null()
        & (pl.col("url") != "")
        & (
            pl.col("url").str.to_lowercase().str.starts_with("http://")
            | pl.col("url").str.to_lowercase().str.starts_with("https://")
        )
    )
)

jumlah_awal = (
    lf.select(pl.len())
    .collect(engine="streaming")
    .item()
)

jumlah_setelah_missing = (
    lf_missing_clean.select(pl.len())
    .collect(engine="streaming")
    .item()
)

print(f"Baris sebelum pembersihan: {jumlah_awal:,}")
print(f"Baris setelah pembersihan: {jumlah_setelah_missing:,}")
print(f"Baris yang dihapus: {jumlah_awal - jumlah_setelah_missing:,}")

Baris sebelum pembersihan: 305,000
Baris setelah pembersihan: 305,000
Baris yang dihapus: 0


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [9]:
# Deteksi URL duplikat
stat_url = (
    lf_missing_clean.select(
        pl.len().alias("jumlah_baris"),
        pl.col("url").n_unique().alias("url_unik"),
    )
    .collect(engine="streaming")
)

baris_sebelum_dedup = stat_url["jumlah_baris"][0]
url_unik = stat_url["url_unik"][0]
jumlah_duplikat_url = baris_sebelum_dedup - url_unik

print(f"Baris sebelum deduplikasi: {baris_sebelum_dedup:,}")
print(f"Jumlah URL unik: {url_unik:,}")
print(f"Duplikasi URL yang ditemukan: {jumlah_duplikat_url:,}")

# Pertahankan satu baris untuk setiap URL
clean_lf = lf_missing_clean.unique(
    subset=["url"],
    keep="first",
)

jumlah_setelah_dedup = (
    clean_lf.select(pl.len())
    .collect(engine="streaming")
    .item()
)

print(f"Baris setelah deduplikasi: {jumlah_setelah_dedup:,}")

Baris sebelum deduplikasi: 305,000
Jumlah URL unik: 305,000
Duplikasi URL yang ditemukan: 0
Baris setelah deduplikasi: 305,000


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [10]:
# Hitung kuartil panjang teks untuk mendeteksi outlier
panjang_teks = pl.col("text").str.len_chars()

kuartil = (
    clean_lf.select(
        panjang_teks.quantile(0.25).alias("Q1"),
        panjang_teks.quantile(0.75).alias("Q3"),
        panjang_teks.min().alias("minimum"),
        panjang_teks.max().alias("maksimum"),
        panjang_teks.mean().alias("rata_rata"),
    )
    .collect(engine="streaming")
)

q1 = kuartil["Q1"][0]
q3 = kuartil["Q3"][0]
iqr = q3 - q1
batas_atas = q3 + 1.5 * iqr

jumlah_outlier = (
    clean_lf.select(
        (panjang_teks > batas_atas)
        .sum()
        .alias("jumlah_outlier")
    )
    .collect(engine="streaming")
    .item()
)

print("Statistik panjang teks:")
display(kuartil)

print(f"Batas atas IQR: {batas_atas:.2f} karakter")
print(f"Jumlah teks di atas batas IQR: {jumlah_outlier:,}")

Statistik panjang teks:


Q1,Q3,minimum,maksimum,rata_rata
f64,f64,u32,u32,f64
1447.0,3295.0,62,185713,3160.015115


Batas atas IQR: 6067.00 karakter
Jumlah teks di atas batas IQR: 24,707


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [11]:
# Simpan hasil bersih secara streaming
clean_lf.sink_parquet(
    str(OUTPUT_PATH),
    compression="zstd",
)

print("Dataset bersih disimpan di:")
print(OUTPUT_PATH)

# Validasi hasil menggunakan DuckDB
output_sql_path = str(OUTPUT_PATH).replace("\\", "/")
output_sql_path = output_sql_path.replace("'", "''")

con = duckdb.connect()

validasi = con.execute(f"""
    SELECT
        COUNT(*) AS jumlah_baris,
        COUNT(DISTINCT url) AS url_unik,
        SUM(
            CASE WHEN text IS NULL OR TRIM(text) = ''
            THEN 1 ELSE 0 END
        ) AS teks_kosong,
        SUM(
            CASE WHEN timestamp IS NULL
            THEN 1 ELSE 0 END
        ) AS timestamp_kosong,
        SUM(
            CASE WHEN url IS NULL OR TRIM(url) = ''
            THEN 1 ELSE 0 END
        ) AS url_kosong
    FROM read_parquet('{output_sql_path}')
""").df()

display(validasi)

Dataset bersih disimpan di:
/home/jovyan/work/data/processed/dataset_bersih.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,jumlah_baris,url_unik,teks_kosong,timestamp_kosong,url_kosong
0,305000,305000,0.0,0.0,0.0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [13]:
# 7. Simpan Data Bersih

# Pastikan folder output tersedia
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Hitung jumlah baris sebelum cleaning
baris_sebelum = (
    lf.select(pl.len())
    .collect(engine="streaming")
    .item()
)

# Simpan hasil cleaning tanpa memuat seluruh hasil ke RAM
clean_lf.sink_parquet(
    str(OUTPUT_PATH),
    compression="zstd",
)

# Hitung jumlah baris sesudah cleaning
baris_sesudah = (
    clean_lf.select(pl.len())
    .collect(engine="streaming")
    .item()
)

print(f"Baris sebelum cleaning: {baris_sebelum:,}")
print(f"Baris sesudah cleaning: {baris_sesudah:,}")
print(f"Baris yang dihapus: {baris_sebelum - baris_sesudah:,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")
print(f"Ukuran file: {OUTPUT_PATH.stat().st_size / 1_000_000:.2f} MB")

Baris sebelum cleaning: 305,000
Baris sesudah cleaning: 305,000
Baris yang dihapus: 0
Data bersih tersimpan di: /home/jovyan/work/data/processed/dataset_bersih.parquet
Ukuran file: 365.38 MB


## Catatan Milestone 2

### 1. Dimensi kualitas data

Dataset yang digunakan adalah mC4 Indonesia dalam format Parquet dengan total 305.000 baris dan 3 kolom, yaitu `text`, `timestamp`, dan `url`.

Profiling dilakukan menggunakan DuckDB `SUMMARIZE` untuk memeriksa tipe data, jumlah nilai, nilai minimum dan maksimum, serta karakteristik setiap kolom.

### 2. Penanganan missing values

Pemeriksaan dilakukan pada kolom `text`, `timestamp`, dan `url`. Nilai kosong dan spasi pada teks diperiksa, timestamp dikonversi ke format tanggal-waktu dengan zona waktu UTC, dan URL divalidasi agar menggunakan protokol HTTP atau HTTPS.

Berdasarkan hasil pemeriksaan awal, tidak ditemukan nilai null pada ketiga kolom. Baris yang tidak memenuhi aturan validasi disaring melalui proses cleaning.

### 3. Duplikasi dan outlier

Pemeriksaan duplikasi dilakukan berdasarkan kolom `url`. Jumlah baris sebelum dan sesudah cleaning tetap 305.000, sehingga proses cleaning tidak mengurangi jumlah baris pada hasil akhir.

Outlier diperiksa berdasarkan panjang teks menggunakan metode Interquartile Range (IQR). Teks panjang tidak otomatis dihapus karena panjang teks saja tidak membuktikan bahwa suatu dokumen tidak valid.

### 4. Hasil cleaning

- Baris sebelum cleaning: 305.000.
- Baris sesudah cleaning: 305.000.
- Baris yang dihapus: 0.
- File hasil cleaning: `data/processed/dataset_bersih.parquet`.
- Ukuran file hasil cleaning: sekitar 365,38 MB.

### 5. Aturan validasi

Dataset bersih perlu memenuhi aturan berikut:
- Kolom `text` tidak null dan tidak kosong setelah spasi dibersihkan.
- Kolom `timestamp` berhasil dikonversi ke format tanggal-waktu.
- Kolom `url` tidak null, tidak kosong, dan menggunakan protokol HTTP atau HTTPS.
- Duplikasi URL diperiksa agar setiap URL hanya dipertahankan satu kali.

Verifikasi akhir dilakukan menggunakan DuckDB terhadap file hasil cleaning untuk memastikan aturan kualitas data tetap terpenuhi.

In [14]:
# Verifikasi akhir dataset bersih menggunakan DuckDB
output_sql_path = str(OUTPUT_PATH).replace("\\", "/")
output_sql_path = output_sql_path.replace("'", "''")

validasi = con.execute(f"""
    SELECT
        COUNT(*) AS jumlah_baris,
        COUNT(DISTINCT url) AS url_unik,
        SUM(
            CASE WHEN text IS NULL OR TRIM(text) = ''
            THEN 1 ELSE 0 END
        ) AS teks_kosong,
        SUM(
            CASE WHEN timestamp IS NULL
            THEN 1 ELSE 0 END
        ) AS timestamp_kosong,
        SUM(
            CASE WHEN url IS NULL OR TRIM(url) = ''
            THEN 1 ELSE 0 END
        ) AS url_kosong,
        SUM(
            CASE WHEN NOT (
                LOWER(url) LIKE 'http://%'
                OR LOWER(url) LIKE 'https://%'
            )
            THEN 1 ELSE 0 END
        ) AS url_tidak_valid
    FROM read_parquet('{output_sql_path}')
""").df()

display(validasi)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,jumlah_baris,url_unik,teks_kosong,timestamp_kosong,url_kosong,url_tidak_valid
0,305000,305000,0.0,0.0,0.0,0.0
